# Chapter 16
## Model Neurons of Bifurcation Type 3
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter16.ipynb)

## About this chapter

Type-3 excitability (onset via a SNIC -- saddle-node on invariant circle --
producing a homoclinic-like slow passage rather than a Hopf) is studied in
a reduced INa,p+IK model and, separately, in a "self-exciting" theta
neuron whose slow adaptation variable $z$ is boosted at each spike,
producing a build-up-and-reset firing pattern rather than steady tonic
firing.

$$
C\dot v=g_{Na}m_\infty(v)(E_{Na}-v)+g_K n(E_K-v)+g_L(E_L-v)+I,\qquad
\tau_n\dot n=n_\infty(v)-n.
$$

See [`README.md`](chapter16.md) for the full guide, including suggested
order and related chapters. The `SETN_PHASE_PLANE` cell near the bottom is
slow (a couple of minutes) -- `dt=1e-4` with some trajectories lingering
near a slow-passage threshold.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from ipywidgets import interact
from mnd.core import draw_arrow

## $I_{Na,p}+I_K$ Fixed-Point Bifurcation Diagram

In [ ]:
def inapik_m_inf(v):
    return 1.0 / (1 + exp((-20 - v) / 15))


def inapik_m_inf_p(v):
    return -1.0 / (1 + exp((-20 - v) / 15)) ** 2 * exp((-20 - v) / 15) * (-1 / 15)


def inapik_n_inf(v):
    return 1.0 / (1 + exp((-25 - v) / 5))


def inapik_n_inf_p(v):
    return -1.0 / (1 + exp((-25 - v) / 5)) ** 2 * exp((-25 - v) / 5) * (-1 / 5)


def inapik_jacobian(v_c, n_c, g_na=20.0, g_k=10.0, g_l=8.0, v_na=60.0, v_k=-90.0, tau_n=0.15):
    j00 = g_na * inapik_m_inf_p(v_c) * (v_na - v_c) - g_na * inapik_m_inf(v_c) - g_k * n_c - g_l
    j01 = g_k * (v_k - v_c)
    j10 = inapik_n_inf_p(v_c) / tau_n
    j11 = -1 / tau_n
    return np.array([[j00, j01], [j10, j11]])


def simulate_inapik_fixed_points(c=1.0, g_na=20.0, g_k=10.0, g_l=8.0,
                                  v_na=60.0, v_k=-90.0, v_l=-80.0, tau_n=0.15,
                                  i_ext_vec=None):
    if i_ext_vec is None:
        i_ext_vec = -4 + np.arange(1001) / 1000 * 12

    def f(v):
        """zeros of this function (offset by I) are the fixed points"""
        return g_na * inapik_m_inf(v) * (v_na - v) + g_k * inapik_n_inf(v) * (v_k - v) + g_l * (v_l - v)

    v_grid = -100 + np.arange(3001) / 3000 * 150
    f_grid = f(v_grid)

    def find_fixed_points(I):
        fi = f_grid + I
        roots = []
        for j in np.where(fi[:-1] * fi[1:] <= 0)[0]:
            v_low, v_high = v_grid[j], v_grid[j + 1]
            while v_high - v_low > 1e-12:
                v_c = (v_low + v_high) / 2
                if (f(v_c) + I) * (f(v_high) + I) <= 0:
                    v_low = v_c
                else:
                    v_high = v_c
            roots.append((v_low + v_high) / 2)
        return roots

    points = {'black': [], 'blue': [], 'red': [], 'green': [], 'magenta': []}
    for I in i_ext_vec:
        for v_c in find_fixed_points(I):
            n_c = inapik_n_inf(v_c)
            e = np.linalg.eigvals(inapik_jacobian(v_c, n_c, g_na, g_k, g_l, v_na, v_k, tau_n))
            if abs(e[0].imag) < 1e-12 and e[0].real < 0 and e[1].real < 0:
                points['black'].append((I, v_c))
            if abs(e[0].imag) < 1e-12 and e[0].real > 0 and e[1].real > 0:
                points['blue'].append((I, v_c))
            if abs(e[0].imag) > 1e-12 and e[0].real < 0:
                points['red'].append((I, v_c))
            if abs(e[0].imag) > 1e-12 and e[0].real > 0:
                points['green'].append((I, v_c))
            if abs(e[0].imag) < 1e-12 and e[0].real * e[1].real < 0:
                points['magenta'].append((I, v_c))
    return points, i_ext_vec


def plot_inapik_fixed_points(points, i_ext_vec):
    all_v = [v for pts in points.values() for _, v in pts]
    maxv_c, minv_c = max(all_v + [-100]), min(all_v + [50])

    plt.figure(figsize=(7, 7))
    for color, key in [('k', 'black'), ('b', 'blue'), ('r', 'red')]:
        if points[key]:
            i_pts, v_pts = zip(*points[key])
            plt.plot(i_pts, v_pts, '.' + color, markersize=8)
    for color, key in [('g', 'green'), ('m', 'magenta')]:
        if points[key]:
            i_pts, v_pts = zip(*points[key])
            plt.plot(i_pts, v_pts, '--' + color, linewidth=4)

    plt.xlabel('$I$')
    plt.ylabel(r'$v_\ast$')
    plt.xlim(-4, 8)
    plt.ylim(minv_c - 5, maxv_c + 5)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_inapik_fixed_points(*simulate_inapik_fixed_points())

## $I_{Na,p}+I_K$ Phase Plane

Four panels ($I=-5,-1.4,2,4.4$) from resting near the SNIC to firing on
the stable limit cycle.

In [ ]:
def inapik_classify_fixed_points(I, c=1.0, g_na=20.0, g_k=10.0, g_l=8.0,
                                  v_na=60.0, v_k=-90.0, v_l=-80.0, tau_n=0.15):
    def f(v):
        return g_na * inapik_m_inf(v) * (v_na - v) + g_k * inapik_n_inf(v) * (v_k - v) + g_l * (v_l - v)

    w_grid = -100 + np.arange(10001) / 10000 * 150
    f_grid = f(w_grid)
    fi = f_grid + I
    roots = []
    for j in np.where(fi[:-1] * fi[1:] <= 0)[0]:
        w_low, w_high = w_grid[j], w_grid[j + 1]
        while w_high - w_low > 1e-12:
            w_c = (w_low + w_high) / 2
            if (f(w_c) + I) * (f(w_high) + I) <= 0:
                w_low = w_c
            else:
                w_high = w_c
        roots.append((w_low + w_high) / 2)

    out = {'black': [], 'blue': [], 'red': [], 'green': [], 'magenta': []}
    for v_c in roots:
        n_c = inapik_n_inf(v_c)
        e = np.linalg.eigvals(inapik_jacobian(v_c, n_c, g_na, g_k, g_l, v_na, v_k, tau_n))
        if abs(e[0].imag) < 1e-12 and e[0].real < 0 and e[1].real < 0:
            out['black'].append((v_c, n_c))
        if abs(e[0].imag) < 1e-12 and e[0].real > 0 and e[1].real > 0:
            out['blue'].append((v_c, n_c))
        if abs(e[0].imag) > 1e-12 and e[0].real < 0:
            out['red'].append((v_c, n_c))
        if abs(e[0].imag) > 1e-12 and e[0].real > 0:
            out['green'].append((v_c, n_c))
        if abs(e[0].imag) < 1e-12 and e[0].real * e[1].real < 0:
            out['magenta'].append((v_c, n_c))
    return out


def inapik_simulate(v0, n0, i_ext, t_final, dt, c=1.0, g_na=20.0, g_k=10.0, g_l=8.0,
                     v_na=60.0, v_k=-90.0, v_l=-80.0, tau_n=0.15):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    v = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    v[0], n[0] = v0, n0

    for k in range(m_steps):
        m = inapik_m_inf(v[k])
        v_inc = (g_na * m * (v_na - v[k]) + g_k * n[k] * (v_k - v[k]) + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = (inapik_n_inf(v[k]) - n[k]) / tau_n

        v_tmp = v[k] + dt05 * v_inc
        m_tmp = inapik_m_inf(v_tmp)
        n_tmp = n[k] + dt05 * n_inc

        v_inc = (g_na * m_tmp * (v_na - v_tmp) + g_k * n_tmp * (v_k - v_tmp) + g_l * (v_l - v_tmp) + i_ext) / c
        n_inc = (inapik_n_inf(v_tmp) - n_tmp) / tau_n

        v[k + 1] = v[k] + dt * v_inc
        n[k + 1] = n[k] + dt * n_inc

    return v, n


def simulate_inapik_phase_plane():
    panels = []

    v1, n1 = inapik_simulate(-50.0, 0.0, i_ext=-5.0, t_final=2.5, dt=0.001)
    panels.append(dict(v_full=v1, n_full=n1, v=v1, n=n1, dt=0.001, i_ext=-5.0, title=r'$I=-5$',
                        arrows=[(0.4, 0.0), (0.7, 0.0), (1.8, 0.002)]))

    v2, n2 = inapik_simulate(-52.5, 0.0, i_ext=-1.4, t_final=3.0, dt=0.001)
    panels.append(dict(v_full=v2, n_full=n2, v=v2, n=n2, dt=0.001, i_ext=-1.4, title=r'$I=-1.4$',
                        arrows=[(0.5, 0.0), (0.9, 0.0)]))

    t_final = 20.0
    dt = 0.001
    v3, n3 = inapik_simulate(-30.0, 0.2, i_ext=2.0, t_final=t_final, dt=dt)
    tail = round((t_final - 1.2) / dt)
    panels.append(dict(v_full=v3, n_full=n3, v=v3[tail:], n=n3[tail:], dt=dt, i_ext=2.0, title=r'$I=2$',
                        arrows=[(t_final - 1.6, 0.0), (t_final - 1.9, 0.0)]))

    v4, n4 = inapik_simulate(-30.0, 0.2, i_ext=4.4, t_final=t_final, dt=dt)
    tail = round((t_final - 2) / dt)
    panels.append(dict(v_full=v4, n_full=n4, v=v4[tail:], n=n4[tail:], dt=dt, i_ext=4.4, title=r'$I=4.4$',
                        arrows=[(t_final - 1, 0.0), (t_final - 1.3, 0.0)]))
    return panels


def plot_inapik_phase_plane(panels):
    A, B, C, D = -75, 0, -0.15, 0.85

    def _plot_fixed_points(ax, fp):
        for v_c, n_c in fp['black']:
            ax.plot(v_c, n_c, '.k', markersize=15)
        for v_c, n_c in fp['blue']:
            ax.plot(v_c, n_c, '.b', markersize=15)
        for v_c, n_c in fp['red']:
            ax.plot(v_c, n_c, '.r', markersize=15)
        for v_c, n_c in fp['green']:
            ax.plot(v_c, n_c, 'og', markersize=6, markerfacecolor='none')
        for v_c, n_c in fp['magenta']:
            ax.plot(v_c, n_c, 'om', markersize=6, markerfacecolor='none')

    def _arrow(ax, v, n, t0, dt, epsilon=0.07, theta=0.0):
        k0 = round(t0 / dt)
        x0, y0 = v[k0], n[k0]
        vec = np.array([(v[k0 + 1] - v[k0 - 1]) / (2 * dt),
                         (n[k0 + 1] - n[k0 - 1]) / (2 * dt)])
        if theta:
            rot = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
            vec = rot @ vec
        draw_arrow(ax, (A, B), (C, D), x0, y0, vec, epsilon=epsilon, width=2, color='k')

    fig, axes = plt.subplots(2, 2, figsize=(9, 9))
    for ax, p in zip(axes.flat, panels):
        ax.plot(p['v'], p['n'], '-k', linewidth=2)
        fp = inapik_classify_fixed_points(p['i_ext'])
        _plot_fixed_points(ax, fp)
        for t0, theta in p['arrows']:
            _arrow(ax, p['v_full'], p['n_full'], t0, p['dt'], theta=theta)
        ax.set_xlim(A, B)
        ax.set_ylim(C, D)
        ax.set_box_aspect(1)
        ax.set_title(p['title'])

    axes[1, 0].set_xlabel('$v$')
    axes[1, 0].set_ylabel('$n$')
    axes[1, 1].set_xlabel('$v$')
    axes[0, 0].set_ylabel('$n$')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_inapik_phase_plane(simulate_inapik_phase_plane())

## Self-Exciting Theta Neuron

Each spike (a $\theta$ wraparound) boosts the slow adaptation variable
$z$ by resetting it to $w_{max}$; between spikes it decays with time
constant $\tau_w$.

In [ ]:
def simulate_self_exciting_theta_neuron(I=-0.05, w_max=0.20, tau_w=20.0, t_final=50.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    theta = np.zeros(m_steps + 1)
    w = np.zeros(m_steps + 1)
    theta[0] = np.pi / 2
    k_spikes = []

    for k in range(m_steps):
        theta_inc = 1 - np.cos(theta[k]) + (I + w[k]) * (1 + np.cos(theta[k]))
        w_inc = -w[k] / tau_w
        theta_tmp = theta[k] + dt05 * theta_inc
        w_tmp = w[k] + dt05 * w_inc
        theta_inc = 1 - np.cos(theta_tmp) + (I + w_tmp) * (1 + np.cos(theta_tmp))
        w_inc = -w_tmp / tau_w
        theta[k + 1] = theta[k] + dt * theta_inc
        w[k + 1] = w[k] + dt * w_inc
        if theta[k + 1] > np.pi:
            theta[k + 1] = -np.pi
            w[k + 1] = w_max
            k_spikes.append(k)

    t = np.arange(m_steps + 1) * dt
    return t, theta, w, k_spikes, w_max


def plot_self_exciting_theta_neuron(t, theta, w, k_spikes, w_max):
    fig, ax = plt.subplots(2, figsize=(7, 6), sharex=True)
    ax[0].plot(t, 1 - np.cos(theta), '-k', linewidth=2)
    ax[0].set_ylim(0, 2)
    ax[0].set_ylabel(r'$1-\cos\theta$')

    ax[1].plot(t[:k_spikes[0] + 1], w[:k_spikes[0] + 1], '-k', linewidth=2)
    t_mid = (t[k_spikes[0]] + t[k_spikes[0] + 1]) / 2
    ax[1].plot([t_mid, t_mid], [w[k_spikes[0]], w_max], ':k', linewidth=2)
    for i in range(len(k_spikes) - 1):
        ax[1].plot(t[k_spikes[i] + 1:k_spikes[i + 1] + 1],
                    w[k_spikes[i] + 1:k_spikes[i + 1] + 1], '-k', linewidth=2)
        t_mid = (t[k_spikes[i + 1]] + t[k_spikes[i + 1] + 1]) / 2
        ax[1].plot([t_mid, t_mid], [w[k_spikes[i + 1]], w_max], ':k', linewidth=2)
    ax[1].plot(t[k_spikes[-1] + 1:], w[k_spikes[-1] + 1:], '-k', linewidth=2)

    ax[1].set_xlim(0, t[-1])
    if w_max > 0:
        ax[1].set_ylim(0, w_max)
    ax[1].set_xlabel('$t$ [ms]')
    ax[1].set_ylabel('$z$')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_self_exciting_theta_neuron(*simulate_self_exciting_theta_neuron())

In [ ]:
interact(lambda I=-0.05, w_max=0.20: plot_self_exciting_theta_neuron(
    *simulate_self_exciting_theta_neuron(I=I, w_max=w_max)),
    I=(-0.2, 0.2, 0.01), w_max=(0.0, 0.4, 0.02));

## Self-Exciting Theta Neuron, Smooth Reset

Replaces the instantaneous reset-to-$w_{max}$ with a smooth,
voltage-dependent kick term added directly to $\dot z$.

In [ ]:
def simulate_self_exciting_theta_smooth(I=-0.05, w_max=0.2, tau_w=20.0, t_final=50.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    theta = np.zeros(m_steps + 1)
    w = np.zeros(m_steps + 1)
    theta[0] = np.pi / 2

    for k in range(m_steps):
        theta_inc = 1 - np.cos(theta[k]) + (I + w[k]) * (1 + np.cos(theta[k]))
        w_inc = -w[k] / tau_w + 10 * np.exp(-5 * (1 + np.cos(theta[k]))) * (w_max - w[k])
        theta_tmp = theta[k] + dt05 * theta_inc
        w_tmp = w[k] + dt05 * w_inc
        theta_inc = 1 - np.cos(theta_tmp) + (I + w_tmp) * (1 + np.cos(theta_tmp))
        w_inc = -w_tmp / tau_w + 10 * np.exp(-5 * (1 + np.cos(theta_tmp))) * (w_max - w_tmp)
        theta[k + 1] = theta[k] + dt * theta_inc
        w[k + 1] = w[k] + dt * w_inc

    t = np.arange(m_steps + 1) * dt
    return t, theta, w, w_max


def plot_self_exciting_theta_smooth(t, theta, w, w_max):
    fig, ax = plt.subplots(2, figsize=(7, 6), sharex=True)
    ax[0].plot(t, 1 - np.cos(theta), '-k', linewidth=2)
    ax[0].set_xlim(0, t[-1])
    ax[0].set_ylim(0, 2)
    ax[0].set_ylabel(r'$1-\cos\theta$')

    ax[1].plot(t, w, '-k', linewidth=2)
    if w_max > 0:
        ax[1].set_ylim(0, w_max)
    ax[1].set_xlabel('$t$ [ms]')
    ax[1].set_ylabel('$z$')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_self_exciting_theta_smooth(*simulate_self_exciting_theta_smooth())

## Self-Exciting Theta Neuron: Phase Plane

Four regimes ($I$ below, near, and above the SNIC-with-adaptation
threshold $I_\ast$, and above the ordinary SNIC $I_c=0$), each showing 25
trajectories started at different initial $z$.

In [ ]:
def setn_simulate(I, w0, tau_w=20.0, dt=0.0001):
    """Heun/RK2 integration (plain floats/lists) of the self-exciting theta
    neuron until theta wraps past pi or w decays to ~0"""
    dt05 = dt / 2
    theta = [-np.pi]
    w = [w0]
    while theta[-1] < np.pi and w[-1] > 1e-6:
        th, wk = theta[-1], w[-1]
        theta_inc = 1 - np.cos(th) + (I + wk) * (1 + np.cos(th))
        w_inc = -wk / tau_w
        theta_tmp = th + dt05 * theta_inc
        w_tmp = wk + dt05 * w_inc
        theta_inc = 1 - np.cos(theta_tmp) + (I + w_tmp) * (1 + np.cos(theta_tmp))
        w_inc = -w_tmp / tau_w
        theta.append(th + dt * theta_inc)
        w.append(wk + dt * w_inc)
    return np.array(theta), np.array(w)


def setn_theta_thresholds(I):
    if I <= 0:
        theta_plus = np.arccos((1 + I) / (1 - I))
    else:
        theta_plus = 10.0
    return theta_plus, -theta_plus


def simulate_setn_phase_plane(w_max=0.2, tau_w=20.0, dt=0.0001):
    panels = [
        dict(I=-0.15, title=r'$I=-0.15<I_\ast$', label='A', w0_boost=1.014,
             arrows=[(10, 'theta_gt', -2, 2), (10, 'w_lt', 0.06, 2),
                     (13, 'theta_gt', 2, 1), (18, 'theta_gt', 0, 1),
                     (5, 'theta_gt', -2, 1)],
             static_arrows=[(-2.0, 0.0, (1.0, 0.0)), (0.0, 0.0, (-1.0, 0.0)), (2.0, 0.0, (1.0, 0.0))]),
        dict(I=-0.1069150434, title=r'$I>I_\ast$ but $I \approx I_\ast$', label='B', w0_boost=None,
             arrows=[(10, 'w_lt', 0.05, 2), (10, 'theta_gt', -1.5, 2), (10, 'theta_gt', 3, 2),
                     (5, 'theta_gt', -2, 1), (5, 'w_lt', 0.02, 1),
                     (15, 'theta_gt', 0, 1), (20, 'theta_gt', 0, 1)],
             static_arrows=[(-2.0, 0.0, (1.0, 0.0)), (0.0, 0.0, (-1.0, 0.0)), (2.0, 0.0, (1.0, 0.0))]),
        dict(I=-0.05, title=r'$I_\ast < I= -0.05 < I_c=0$', label='C', w0_boost=None,
             arrows=[(10, 'theta_gt', 0, 2),
                     (5, 'theta_gt', -2, 1), (6, 'theta_gt', 2, 1),
                     (15, 'theta_gt', 0, 1), (20, 'theta_gt', 0, 1)],
             static_arrows='C'),
        dict(I=0.05, title=r'$I=0.05>I_c=0$', label='D', w0_boost=None,
             arrows=[(10, 'theta_gt', 0, 2),
                     (5, 'theta_gt', 0, 1), (15, 'theta_gt', 0, 1), (20, 'theta_gt', 0, 1)],
             static_arrows=None),
    ]

    for p in panels:
        trajs = []
        for ijk in range(1, 26):
            w0 = w_max * ijk / 20 * 2
            if p['w0_boost'] and ijk > 10:
                w0 *= p['w0_boost']
            trajs.append(setn_simulate(p['I'], w0, tau_w, dt))
        p['trajs'] = trajs
    return panels


def plot_setn_phase_plane(panels):
    A, B, C, D = -np.pi, np.pi, 0.0, 2 * 0.2

    def _arrow(ax, theta, w, ind, width):
        x, y = theta[ind], w[ind]
        vec = np.array([theta[ind + 1] - theta[ind], w[ind + 1] - w[ind]])
        draw_arrow(ax, (A, B), (C, D), x, y, vec, epsilon=0.06, width=width, color='k')

    def _first(mask):
        return int(np.argmax(mask)) if mask.any() else None

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))

    for ax, p in zip(axes.flat, panels):
        theta_plus, theta_minus = setn_theta_thresholds(p['I'])
        ax.plot(theta_plus, 0, 'ok', markersize=6, markerfacecolor='w')
        ax.plot(theta_minus, 0, 'ok', markersize=6, markerfacecolor='k')

        for ijk, (theta, w) in enumerate(p['trajs'], start=1):
            ax.plot(theta, w, '-k', linewidth=3 if ijk == 10 else 1)

        for ijk, cond, thresh, width in p['arrows']:
            theta, w = p['trajs'][ijk - 1]
            mask = theta > thresh if cond == 'theta_gt' else w < thresh
            ind = _first(mask)
            _arrow(ax, theta, w, ind, width)

        if p['static_arrows'] == 'C':
            static = [(-2.0, 0.0, (1.0, 0.0)), (-theta_plus / 3, 0.0, (-1.0, 0.0)), (2.0, 0.0, (1.0, 0.0))]
        else:
            static = p['static_arrows']
        if static:
            for x, y, vec in static:
                draw_arrow(ax, (A, B), (C, D), x, y, np.array(vec), epsilon=0.06, width=1, color='k')

        ax.set_xlim(A, B)
        ax.set_ylim(C, D)
        ax.set_box_aspect(1)
        ax.set_title(p['title'])
        ax.text(-np.pi - 0.9, 0.4, p['label'], fontsize=16, fontweight='bold')

    axes[1, 0].set_xlabel(r'$\theta$')
    axes[1, 0].set_ylabel('$z$')
    axes[1, 1].set_xlabel(r'$\theta$')
    axes[0, 0].set_ylabel('$z$')

    plt.tight_layout()
    plt.show()

In [ ]:
# Slow (a couple of minutes): 25 trajectories x 4 panels at dt=1e-4.
plot_setn_phase_plane(simulate_setn_phase_plane())